# University Policy Assistant — RAG-Powered LLM System

## Problem Definition

**Domain:** Higher Education — University Policy Information Retrieval

**The Problem:**
Universities generate and maintain vast amounts of policy documents — handbooks, regulations, procedures, and guidelines — stored across multiple formats (text, PDF). Students and staff spend significant time searching through fragmented documents to find answers to policy questions (e.g., attendance requirements, grading rules, appeals processes, extenuating circumstances). This search friction leads to:
- Increased administrative burden on staff
- Slower decision-making for students
- Misinterpretation or non-compliance with policies
- Higher frustration due to lack of clear, timely answers

**The Solution:**
An LLM-powered Retrieval-Augmented Generation (RAG) system that answers policy questions by:
1. Retrieving relevant policy passages from a knowledge base (embeddings + vector search)
2. Generating accurate, context-grounded answers using a lightweight LLM
3. Providing source document citations for transparency
4. Applying guardrails to ensure safe and appropriate use

**Target Users:** University students seeking policy information across attendance, grading, academic integrity, conduct, extenuating circumstances, appeals, data protection, library resources, IT use, disability support, and tuition fees.

**Course Theory Reference:** This project applies concepts from:
- **W5-L2 (Prompt Engineering & RAG):** Retrieval-Augmented Generation architecture
- **W5-lab2 (Advanced RAG Architectures):** Vector search, FAISS, chunking strategies
- **W4-L1/W4-L2 (LLM Foundations):** Transformer-based language models, FLAN-T5
- **W6-L2 (Deployment & Ethics):** Responsible AI, safeguards, evaluation

## Architecture & Design Choices

### System Architecture Diagram

```
┌─────────────────────────────────────────────────────────────┐
│                   USER INTERFACE (Colab)                    │
│         Query Input → Display Answer + Sources             │
└───────────────────────────┬─────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────┐
│                   SAFEGUARD (Layer 1)                       │
│         Input Keyword Filter — blocks unsafe queries        │
└───────────────────────────┬─────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────┐
│                   EMBEDDING MODEL                           │
│         all-MiniLM-L6-v2 (sentence-transformers)            │
│         Converts query → 384-dim vector                    │
└───────────────────────────┬─────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────┐
│                   VECTOR DATABASE (FAISS)                   │
│         IndexFlatL2 — L2 distance search                    │
│         Returns top-k (k=3) relevant chunks                │
└───────────────────────────┬─────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────┐
│                   LLM GENERATOR (FLAN-T5-base)              │
│         Prompt Template + Retrieved Context → Answer        │
│         Safeguard (Layer 2): Prompt-level guardrails       │
└───────────────────────────┬─────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────┐
│                   OUTPUT                                   │
│         AI Answer + Source Document Citations               │
└─────────────────────────────────────────────────────────────┘
```

### Design Choices

| Component | Choice | Why | Alternative | Why Not |
|-----------|--------|-----|-------------|---------|
| Embeddings | all-MiniLM-L6-v2 | Lightweight (80MB), good semantic quality, Colab-friendly, widely used in course | BERT-large (340MB) | Too large for Colab free; slower inference |
| Vector DB | FAISS (IndexFlatL2) | In-memory, fast L2 search, no cloud dependency, taught in W5-lab2 | Pinecone/Chroma | Requires API keys, cloud dependency |
| LLM | FLAN-T5-base (250M params) | Lightweight, text2text, HF pipeline, taught in W4-L1, Colab-viable | Llama 7B | Won't fit Colab free tier without quantization |
| Chunking | RecursiveCharacterTextSplitter, 500/50 | Balances context with retrieval precision, taught in class | Fixed 200 chars | Too short for policy paragraphs |
| Documents | .txt + .pdf | Dual format demonstrates flexibility, PDFs are real-world format | .txt only | Less realistic; PDF is standard for university policies |
| Safeguard | Input filter + prompt guardrails | Simple, no API dependency, meets assignment requirement | Moderation API | Adds cost and external dependency |
| Environment | Google Colab | Free GPU/CPU runtime, zero setup, reproducible | Local machine | Inconsistent environments across evaluators |

## Step 1: Environment Setup

**What this does:** Installs all required Python packages for the RAG pipeline.

**Why this matters:** Colab environments do not have NLP/ML libraries pre-installed. We need sentence-transformers for embeddings, FAISS for vector search, transformers for the LLM, and langchain for orchestration.

**Key packages:**
- `sentence-transformers`: Embedding model (all-MiniLM-L6-v2)
- `faiss-cpu`: Vector similarity search (Facebook AI Similarity Search)
- `transformers`: HuggingFace model hub, FLAN-T5 pipeline
- `langchain` / `langchain-community`: Document loading, chunking, RAG chain orchestration
- `pypdf`: PDF document loading (for .pdf policy files)
- `numpy` / `pandas`: Data handling and evaluation tables

**Trade-off:** We install CPU versions of FAISS and PyTorch. GPU acceleration would speed up embeddings and LLM inference, but CPU is sufficient for this small-scale prototype and ensures maximum Colab compatibility.

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers pypdf numpy pandas langchain-community langchain-text-splitters
print("Dependencies installed successfully")


## Step 2: Data Handling — Load Knowledge Base

**What this does:** Loads policy documents from the KnowledgeBase directory, supporting both .txt and .pdf formats.

**Data Description:** The KnowledgeBase contains 11 university policy documents covering:
- **Existing (.txt):** Academic Integrity, Assessment & Grading, Attendance Policy, Code of Conduct, Extenuating Circumstances
- **New (.pdf):** Student Appeals, Data Protection & Privacy, Library Resources, IT Acceptable Use, Disability Support, Tuition & Fees

**Why two formats?** Real-world university policy documents come in both formats. .txt files are simple and easy to create; .pdf files represent the standard distribution format for official university policies. Supporting both demonstrates practical data handling skills.

**Limitations:** This is a small curated corpus (11 documents). A production system would need ingestion pipelines for hundreds of documents, automated updates when policies change, and version tracking.

**Colab Note:** The code below uses a local path. In Colab, you will upload files or mount Google Drive (Cell 23 covers this).

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from typing import List
from pathlib import Path

# Load .txt files using LangChain's DirectoryLoader
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader

# Configure path — UPDATE THIS for your environment
# KB_PATH = "/c/Users/user/Downloads/SLIIT/MCS Module 2/Assignment/App/APP/KnowledgeBase"
KB_PATH = str(Path.cwd() / "KnowledgeBase")

print("--- Loading Knowledge Base ---")

# Load .txt files
txt_loader = DirectoryLoader(KB_PATH, glob="*.txt", loader_cls=TextLoader)
txt_docs = txt_loader.load()
print(f"Loaded {len(txt_docs)} .txt documents")

# Load .pdf files
pdf_files = glob.glob(os.path.join(KB_PATH, "*.pdf"))
pdf_docs = []
for pdf_path in pdf_files:
    loader = PyPDFLoader(pdf_path)
    pdf_docs.extend(loader.load())
print(f"Loaded {len(pdf_files)} .pdf files ({len(pdf_docs)} pages)")

# Combine all documents
documents = txt_docs + pdf_docs
print(f"Total documents: {len(documents)}")

# Show document sources
doc_sources = list(set([doc.metadata.get('source', 'unknown') for doc in documents]))
print("\nKnowledge Base contents:")
for s in sorted(doc_sources):
    print(f"  - {os.path.basename(s)}")

## Step 3: Document Chunking

**What this does:** Splits large policy documents into smaller, overlapping chunks suitable for embedding and retrieval.

**Why chunking matters:** LLMs have limited context windows. FLAN-T5-base supports up to 512 tokens. By chunking documents into ~500-character segments with 50-character overlap, we ensure:
1. Each chunk fits comfortably within the model's context window
2. The overlap preserves context across chunk boundaries (e.g., a sentence split mid-way)
3. Retrieved chunks are granular enough to answer specific policy questions precisely

**Chunking strategy:** RecursiveCharacterTextSplitter attempts to split at natural boundaries (paragraphs → sentences → words) before falling back to character-level splitting. This is more intelligent than fixed-size chunking because it respects text structure.

**Trade-off:**
- 500 chars: Good balance between context richness and retrieval precision
- Smaller chunks (200 chars): More precise retrieval but loses surrounding context
- Larger chunks (1000 chars): More context but may include irrelevant information
- Overlap 50 chars: Standard choice; prevents mid-sentence splits from losing meaning

In [ ]:
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

texts = text_splitter.split_documents(documents)

print(f"Documents split into {len(texts)} chunks")

# Statistics
chunk_lengths = [len(c.page_content) for c in texts]
print(f"Average chunk length: {np.mean(chunk_lengths):.0f} chars")
print(f"Min chunk length: {min(chunk_lengths)} chars")
print(f"Max chunk length: {max(chunk_lengths)} chars")
print(f"Std deviation: {np.std(chunk_lengths):.0f} chars")

# Show sample chunks
print("\n--- Sample Chunks ---")
for i, chunk in enumerate(texts[:3]):
    src = chunk.metadata.get('source', 'unknown')
    print(f"\nChunk {i+1} (from {os.path.basename(src)}):")
    print(chunk.page_content[:200] + "...")

## Step 4: Create Embeddings & Vector Store

**What this does:** Converts each text chunk into a dense vector representation (embedding) and stores them in a FAISS index for efficient similarity search.

**How embeddings work:** Sentence-transformers map text into a 384-dimensional vector space where semantically similar texts are positioned close together. When a user submits a query, it is embedded using the same model, and FAISS finds the nearest chunks by L2 (Euclidean) distance.

**Why all-MiniLM-L6-v2?**
- Compact (80MB) — fast to download and run on Colab CPU
- Outputs 384-dim vectors — good balance of speed and accuracy
- Widely used in production RAG systems
- Pre-trained on 1B+ sentence pairs for general semantic understanding

**Why FAISS (IndexFlatL2)?**
- Exact search (not approximate) — guarantees best matches
- In-memory — no server/cloud dependency
- Simple L2 distance — appropriate for small-to-medium corpora
- For larger corpora (100k+ docs), we would switch to IndexIVFFlat for faster search

**Trade-off:** IndexFlatL2 performs exhaustive search (O(n) per query). For 11 documents (~50-80 chunks), this is instantaneous. But for million-scale corpora, approximate nearest neighbour (ANN) indices like HNSW or IVF would be necessary.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("--- Loading Embedding Model ---")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# Quick test
test_embed = embeddings.embed_query("What is the attendance policy?")
print(f"Embedding dimension: {len(test_embed)}")
print(f"First 5 values: {test_embed[:5]}")

print("\n--- Creating FAISS Vector Store ---")
vector_db = FAISS.from_documents(texts, embeddings)
print(f"Vector store created with {vector_db.index.ntotal} vectors")

# Quick retrieval test
test_query = "What is the attendance requirement?"
test_results = vector_db.similarity_search(test_query, k=3)
print(f"\nRetrieval test for: '{test_query}'")
for i, doc in enumerate(test_results):
    src = doc.metadata.get('source', 'unknown')
    print(f"  [{i+1}] from {os.path.basename(src)}: {doc.page_content[:80]}...")

## Step 5: Load LLM — FLAN-T5-base

**What this does:** Loads Google's FLAN-T5-base model via HuggingFace transformers pipeline for text generation.

**Why FLAN-T5-base?**
- **Size:** ~250M parameters (~990MB) — fits comfortably in Colab's free tier RAM/disk
- **Architecture:** Text-to-text transformer — both input and output are text, making it ideal for Q&A
- **Instruction-tuned:** Fine-tuned on a large collection of tasks described via instructions (FLAN = Fine-tuned LAnguage Net)
- **Course relevance:** Covered in W4-L1 (Transformer models, HuggingFace pipelines)
- **No API key needed:** Fully open-source, runs locally

**Why not a larger model?**
- Llama 3 7B (7B params = 28x larger) exceeds Colab free tier memory
- GPT-family models require paid API keys
- DistilGPT2 is an option but lacks instruction-following capability

**Decoding parameters:**
- `temperature=0.3`: Low temperature for factual, deterministic answers (not creative)
- `do_sample=False`: Greedy decoding — always picks the most probable token
- `max_length=256`: Generates answers up to ~200 words, sufficient for policy Q&A
- These choices prioritise factual accuracy over fluency, which is critical for policy responses

In [ ]:
from transformers import pipeline
import torch

print("--- Loading FLAN-T5-base ---")
llm = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_length=256,
    temperature=0.3,
    do_sample=True,
    device=-1  # use CPU (set to 0 for GPU if available)
)

# Test the LLM standalone
test_prompt = "Answer this question: What is the minimum attendance requirement for students?"
result = llm(test_prompt)[0]['generated_text']
print(f"Test response: {result}")

## Step 6: Prompt Template & RetrievalQA Chain

**What this does:** Creates a structured prompt template that instructs the LLM to answer based ONLY on retrieved context, and connects it into a LangChain RetrievalQA chain.

**Prompt design principles:**
1. **Role specification:** "You are a University Policy Assistant" — sets the persona
2. **Grounding instruction:** "Use ONLY the context below" — prevents hallucination from model's internal knowledge
3. **Refusal instruction:** "If the context does not contain the answer, say you don't know" — prevents fabricated answers
4. **Citation prompt:** "Reference the policy document if possible" — supports transparency
5. **Safeguard (Layer 2):** "If the question is not related to university policies, refuse politely" — prompt-level guardrails

**Chain type: "stuff"** — all retrieved documents are "stuffed" into the prompt as context. This is the simplest and fastest approach. For this small corpus (~50-80 chunks, 3 retrieved = ~1500 chars context), it fits well within FLAN-T5's 512-token window.

**Alternative chain types:**
- `map_reduce`: Summarize each doc separately, then combine — better for long documents but slower
- `refine`: Iteratively refine answer with each document — most thorough but highest latency
- `stuff`: Best for our use case — fast, simple, sufficient context

In [ ]:
!pip install -U langchain langchain-core langchain-community

In [ ]:


# Define the prompt template with built-in guardrails
prompt_template = """
You are a University Policy Assistant. Your role is to answer questions about university policies, regulations, and procedures.

INSTRUCTIONS:
- Use ONLY the context below to answer the question.
- If the context does not contain the answer, say "I cannot find this information in the available policy documents."
- If the question is not related to university policies, respond: "I can only answer questions about university policies."
- Reference the policy document name in your answer when possible.
- Keep your answer concise and factual.
- Do not speculate or add information beyond what is in the context.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

print("Prompt template created successfully")
print(PROMPT.template[:300] + "...")

In [ ]:
# Direct RAG pipeline (no LangChain dependency)
def retrieve_context(query, k=3):
    """Retrieve top-k relevant chunks from FAISS."""
    docs = vector_db.similarity_search(query, k=k)
    context = "\n\n---\n\n".join([d.page_content for d in docs])
    sources = list(set([os.path.basename(d.metadata.get('source', 'unknown')) for d in docs]))
    return context, sources

def rag_answer(query):
    """Complete RAG pipeline: retrieve + generate."""
    context, sources = retrieve_context(query)
    
    prompt = f"""
You are a University Policy Assistant. Your role is to answer questions about university policies, regulations, and procedures.

INSTRUCTIONS:
- Use ONLY the context below to answer the question.
- If the context does not contain the answer, say "I cannot find this information in the available policy documents."
- If the question is not related to university policies, respond: "I can only answer questions about university policies."
- Reference the policy document name in your answer when possible.
- Keep your answer concise and factual.
- Do not speculate or add information beyond what is in the context.

CONTEXT:
{context}

QUESTION:
{query}

ANSWER:
"""
    
    result = llm(prompt)[0]['generated_text']
    return {
        "result": result,
        "source_documents": sources
    }

print("RAG pipeline created successfully")
print(f"  Embeddings: all-MiniLM-L6-v2")
print(f"  Vector DB: FAISS IndexFlatL2")
print(f"  Retriever k: 3")
print(f"  LLM: FLAN-T5-base")
print(f"  Prompt: custom template with guardrails")


## Step 7: Safeguard Implementation (≥1 Required)

**What this does:** Implements a two-layer safety system to prevent misuse of the policy assistant.

**Layer 1 — Input Keyword Filter:**
Blocks queries containing keywords associated with academic dishonesty, illegal activity, or attempts to misuse the system. This is checked BEFORE the query reaches the RAG pipeline.

**Layer 2 — Prompt-Level Guardrail:**
Embedded directly in the prompt template (Step 6): The LLM is instructed to refuse off-topic or non-policy questions. Since this is part of the prompt itself, the model will naturally refuse inappropriate queries.

**Why these layers?**
- Simple to implement — no external API calls or additional services
- Directly addresses the assignment requirement for ≥1 safeguard
- Layered approach: filter catches explicit violations, prompt guardrails catch semantic ones

**Limitations:**
- Keyword filtering can be bypassed with synonyms or paraphrasing
- Prompt guardrails are imperfect — a determined user could use prompt injection

**Future improvements:**
- Add a moderation API (e.g., Google Cloud Natural Language API)
- Implement an LLM-as-judge for output verification
- Add rate limiting and user authentication

In [ ]:
# Layer 1: Input keyword filter
UNSAFE_KEYWORDS = [
    "hack", "cheat", "plagiarize", "bypass", "illegal",
    "fake documents", "forged", "bribe", "unauthorised access",
    "malware", "virus", "exploit"
]

def input_safe(query: str) -> bool:
    """Check if a query passes the input safety filter."""
    q = query.lower()
    for kw in UNSAFE_KEYWORDS:
        if kw in q:
            return False
    return True

def safe_generate(query: str) -> dict:
    """Run the RAG pipeline with safety checks."""
    # Layer 1: Input filter
    if not input_safe(query):
        return {
            "result": "⚠️ Query blocked: This query contains unsafe content and cannot be processed.",
            "source_documents": []
        }
    
    # Run RAG (Layer 2 guardrails are in the prompt template)
    result = rag_answer(query)
    return result

# Test the safeguard
safe_queries = [
    "What is the attendance requirement?",
    "How can I hack the grading system?",
    "What is the penalty for plagiarism?",
    "Tell me how to cheat on my exams"
]

for q in safe_queries:
    result = safe_generate(q)
    print(f"Q: {q}")
    print(f"A: {result['result'][:100]}...")
    print()


## Step 8: Testing the Complete RAG Pipeline

**What this does:** Runs the full RAG pipeline on diverse policy queries to verify it works end-to-end.

**Test queries cover all 11 policy documents:** attendance, grading, academic integrity, conduct, extenuating circumstances, appeals, data protection, library, IT use, disability support, tuition fees.

**What to expect:**
- Answers should be grounded in the retrieved policy documents
- Source document names should be visible
- Unsafe queries should be blocked by the safeguard
- Off-topic queries should trigger the prompt-level refusal

In [ ]:
test_queries = [
    "What is the minimum attendance requirement?",
    "How do I apply for extenuating circumstances?",
    "What is the penalty for a Level 2 academic offense?",
    "How long do I have to submit an academic appeal?",
    "What is the grading scale?",
    "How can I request a disability accommodation?",
    "What are the tuition fee payment deadlines?",
    "What is the library borrowing limit for undergraduates?",
    "What is the university's data retention policy?",
    "Can I use university IT for personal purposes?"
]

print("=" * 80)
print("TESTING RAG PIPELINE — 10 POLICY QUERIES")
print("=" * 80)

for i, q in enumerate(test_queries, 1):
    result = safe_generate(q)
    print(f"\n--- Query {i}: {q} ---")
    print(f"Answer: {result['result']}")
    
    # Show source documents
    if 'source_documents' in result and result['source_documents']:
        sources = list(set([
            os.path.basename(d.metadata.get('source', 'unknown'))
            for d in result['source_documents']
        ]))
        print(f"Sources: {', '.join(sources)}")
    print()

## Step 9: Evaluation

**What this does:** Evaluates the RAG system on three dimensions: latency, retrieval quality, and hallucination risk.

### 9.1 Latency Measurement

**Why measure latency?** Response time is a critical UX metric. The system should provide answers within a reasonable time (ideally <10s for a good user experience). Latency depends on:
- Embedding model inference time (~100-200ms per query)
- FAISS search time (~1-10ms for small corpus)
- FLAN-T5 generation time (~1-5s depending on answer length)

**Expected results:** Most time will be spent on LLM generation (the bottleneck), not retrieval.

In [ ]:
import time

eval_queries = [
    "What is the attendance requirement?",
    "How do I apply for extenuating circumstances?",
    "What is the penalty for academic dishonesty?",
    "What is the grading scale for undergraduate courses?",
    "How long do I have to submit an academic appeal?"
]

latencies = []
print("Latency Measurement:")
print("-" * 60)

for q in eval_queries:
    start = time.time()
    result = safe_generate(q)
    elapsed = time.time() - start
    latencies.append({
        "query": q,
        "latency_s": round(elapsed, 2),
        "latency_ms": round(elapsed * 1000, 0)
    })
    print(f"  {q[:50]:50s} {elapsed:.2f}s")

df_latency = pd.DataFrame(latencies)
print("\nLatency Summary:")
print(df_latency.to_string(index=False))
print(f"\nAverage latency: {df_latency['latency_s'].mean():.2f}s")
print(f"Min latency: {df_latency['latency_s'].min():.2f}s")
print(f"Max latency: {df_latency['latency_s'].max():.2f}s")

### 9.2 Retrieval Quality Inspection

**Why inspect retrieval?** The quality of the RAG answer depends entirely on whether the right policy chunks are retrieved. If retrieval fails, the LLM cannot produce a correct answer. This inspection shows exactly what chunks were retrieved for each query.

In [ ]:
print("Retrieval Quality Inspection:")
print("=" * 80)

for q in eval_queries:
    print(f"\nQuery: {q}")
    retrieved_docs = vector_db.similarity_search(q, k=3)
    for i, doc in enumerate(retrieved_docs, 1):
        src = os.path.basename(doc.metadata.get('source', 'unknown'))
        print(f"  [{i}] {src}")
        print(f"       {doc.page_content[:120].strip()}...")

### 9.3 Hallucination Risk Analysis

**Why compare with/without RAG?** This is the most important evaluation — it demonstrates WHY RAG is necessary. A standalone LLM (without retrieval) may produce plausible-sounding but incorrect policy answers because it relies on its training data, which may not have specific university policy information. RAG grounds the answer in verifiable source documents.

**Expected result:** Answers WITHOUT RAG should be generic, potentially incorrect, or hallucinated. Answers WITH RAG should reference specific policy documents and contain verifiable information.

In [ ]:
def answer_without_rag(query):
    """Generate answer without any retrieved context (standalone LLM)."""
    prompt = f"Answer this question about university policy: {query}"
    return llm(prompt)[0]['generated_text']

print("Hallucination Risk: WITH vs WITHOUT RAG")
print("=" * 80)

for q in eval_queries[:3]:  # Test 3 representative queries
    print(f"\nQuery: {q}")
    
    # Without RAG
    no_rag = answer_without_rag(q)
    print(f"  WITHOUT RAG: {no_rag}")
    
    # With RAG
    with_rag = safe_generate(q)
    print(f"  WITH RAG:    {with_rag['result'][:150]}")
    
    # Check for hallucination indicators
    if 'cannot find' in with_rag['result'].lower():
        print(f"  NOTE: RAG could not find information for this query")
    print()

print("\nAnalysis:")
print("- Without RAG: The LLM guesses based on training data — may be wrong")
print("- With RAG: The answer is grounded in retrieved policy documents")
print("- RAG significantly reduces hallucination risk for knowledge-intensive queries")

## Step 10: Running in Google Colab

**What this does:** Provides file upload cells so the notebook can run in Google Colab without local file paths.

**Two options for Colab:**
1. **Upload files directly** — Use `google.colab.files.upload()` to upload individual files
2. **Mount Google Drive** — Mount Drive and point KB_PATH to the folder

**Recommendation:** Option 2 (Drive mount) is more practical for a 11-file knowledge base. Create a folder in your Drive with all policy files, mount it, and update KB_PATH.

**Note:** If you are running this notebook locally (not in Colab), skip these cells and use the local path from Step 2.

In [ ]:
# Option 1: Upload files directly (for Colab)
# from google.colab import files
# uploaded = files.upload()
# This will prompt you to select files from your computer

# Option 2: Mount Google Drive (recommended for multi-file KB)
# from google.colab import drive
# drive.mount('/content/drive')
# Then update KB_PATH to point to your Drive folder

print("Colab setup options are commented out above.")
print("Uncomment the relevant option and update KB_PATH before running.")

## Step 11: Trade-offs Analysis

**What this does:** Summarises the key design trade-offs made in building this system. This analysis is essential for the technical report and demonstrates critical thinking about engineering decisions.

| Trade-off | Choice Made | Benefit | Cost |
|-----------|-------------|---------|------|
| Model size | FLAN-T5-base (250M) | Fits Colab free tier, fast inference | Less capable than 7B+ models for complex reasoning |
| Chunk size | 500 chars | Covers full policy paragraphs | May miss relationships across chunks |
| Embedding model | all-MiniLM-L6-v2 | Fast, small (80MB), good quality | Less accurate than Instructor-XL (1.5GB) |
| Retrieval k | 3 | Enough context for concise answers | May miss relevant docs for multi-part queries |
| Search type | IndexFlatL2 (exact) | Guarantees best matches | Slower than IVF for 100k+ documents |
| Chain type | Stuff | Simple, fast, single LLM call | Context window limits |
| Guardrails | Keyword filter + prompt | No external dependencies | Bypassable with synonyms |
| Document formats | .txt + .pdf | Real-world variety, demonstrates skill | PDF parsing can be imperfect |

**Key insight:** Every choice favours simplicity, reproducibility, and Colab compatibility over maximum performance. This is appropriate for a prototype. A production system would use larger models, hybrid search, and more sophisticated guardrails.

## Step 12: Generate Requirements & Cleanup

**What this does:** Saves requirements.txt and confirms the notebook is complete.

In [ ]:
%%writefile requirements.txt
sentence-transformers==2.2.2
faiss-cpu==1.7.4
transformers==4.36.0
langchain==0.1.0
langchain-community==0.1.0
pypdf==3.17.0
numpy==1.24.0
pandas==2.0.0

print("requirements.txt generated")

## Step 13: Risks, Limitations & Future Work

### Current Limitations

| Limitation | Impact | Mitigation |
|------------|--------|------------|
| Small knowledge base (11 docs) | Limited coverage of policy topics | Expand with more documents from real university handbooks |
| FLAN-T5-base capabilities | May struggle with complex multi-step reasoning | Use larger model (Mistral 7B with 4-bit quantization) |
| Basic keyword safeguard | Can be bypassed with synonyms | Add semantic moderation or LLM-as-judge |
| No PDF table extraction | Tables in policy docs may be missed | Add Camelot or Tabula for table parsing |
| Single-chain RAG (no hybrid) | Misses lexical matches that BM25 would catch | Add hybrid search (BM25 + embeddings + RRF) as in W5-lab2 |
| No caching | Every query re-embeds and regenerates | Add query caching for repeated questions |
| English only | Cannot serve international students | Add multilingual embeddings and LLM (e.g., mT5, BLOOM) |

### Key Risks

1. **Factual accuracy:** Even with RAG, the LLM may misinterpret or misrepresent policy text. Always recommend users verify critical information with official sources.
2. **Data privacy:** Policy documents do not contain personal data in this prototype, but a real deployment must ensure no PII leaks into the vector store.
3. **Over-reliance:** Users may trust AI answers without verification. The UI should display prominent warnings.
4. **Model bias:** FLAN-T5 may reflect biases in its training data. Regular evaluation of outputs for fairness is needed.

### Future Work

1. **PDF ingestion pipeline:** Automate extraction from real university PDF handbooks with table support
2. **Hybrid search:** Add BM25 lexical search + RRF fusion (as demonstrated in W5-lab2)
3. **Conversational memory:** Allow follow-up questions within a session
4. **Streamlit deployment:** Package as a standalone web app (as prototyped in the original app.py)
5. **Hugging Face Spaces:** Deploy for public access and demonstration
6. **Evaluation metrics:** Add BLEU/ROUGE scores, faithfulness metrics, and user satisfaction surveys
7. **Multi-language support:** Expand to support international students

---

## Summary

### What was built

| Requirement | Delivered |
|-------------|-----------|
| Pre-trained model (HuggingFace) | FLAN-T5-base (text2text-generation pipeline) |
| Augmentation technique | RAG — FAISS vector store + embeddings |
| Deployment layer | Google Colab (reproducible notebook) |
| ≥1 safeguard | Input keyword filter + prompt-level guardrails |
| Evaluation | Latency, retrieval quality, hallucination risk comparison |
| Multi-format ingestion | .txt (DirectoryLoader) + .pdf (PyPDFLoader) |
| Source code | Complete .ipynb with 31 cells |
| README | Included |
| requirements.txt | Included |
| Colab-compatible | Yes — pip install + file upload/Drive mount |
| No secrets/keys | All open-source models, no API dependencies |

### Architecture

```
User Query → Input Filter → Embedding (all-MiniLM-L6-v2) → FAISS Search (k=3) →
    Context + Prompt → FLAN-T5-base → Answer + Source Citations
```

### Resources Used

- **Knowledge Base:** 11 policy documents (5 .txt + 6 .pdf) from the University of Excellence
- **Embedding Model:** sentence-transformers/all-MiniLM-L6-v2 (80MB)
- **Vector DB:** FAISS IndexFlatL2 (50-80 chunks total)
- **LLM:** google/flan-t5-base (250M params, ~990MB)

### References to Course Material

- **W5-L2 (Prompt Engineering & RAG):** RAG architecture, prompt design
- **W5-lab2 (Advanced RAG):** Vector search, FAISS, chunking strategies
- **W4-L1/W4-L2 (LLM Foundations):** Transformer models, HuggingFace pipelines, FLAN-T5
- **W6-L2 (Deployment & Ethics):** Responsible AI, safeguards, evaluation methods
- **Lab 2 (NLP Preprocessing):** Text normalization, tokenization concepts
- **Week 2 Labs (ML for NLP):** Evaluation metrics, precision/recall/F1